In [1]:
from torch.nn import functional as F


In [2]:
import torch
import torch.nn as nn
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-10-15 06:00:51--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-15 06:00:51 (30.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt','r',encoding='utf-8') as f:
  text = f.read()

print("Length of dataset in characters: ",len(text))
print(text[:1000])

Length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hung

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s:  [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


In [6]:
# Converting text into a pytorch tensor
import torch
data = torch.tensor(encode(text),dtype=torch.long) #makes the elements 64 bit integers
print(data.shape,data.dtype)
# print(decode(data[:100].tolist()))

torch.Size([1115394]) torch.int64


In [7]:
n = int(0.9*len(data))
train_data = data[:n] #data used to calculate loss and update the model's weight
val_data = data[n:]#used to check how well the model predicts text it wasn't trained on

In [8]:
block_size=8 #model uses 8 characters of context at onced
#  at every position
x = train_data[:block_size]
y = train_data[1: block_size+1]

for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"When input is {context} the target: {target}")

When input is tensor([18]) the target: 47
When input is tensor([18, 47]) the target: 56
When input is tensor([18, 47, 56]) the target: 57
When input is tensor([18, 47, 56, 57]) the target: 58
When input is tensor([18, 47, 56, 57, 58]) the target: 1
When input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [9]:
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Batching
batch_size =4  #four seq processed together
block_size = 8 #each seq contains eight input characters

# B = Batch dimension (How many seq are processed together , here4)..
# T = Time dimension = how many character positions are in each sequence, here,8.
def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)-block_size,(batch_size,)) #chooses four random starting positions

  x = torch.stack([data[i:i + block_size] for i in ix])
  y = torch.stack([data[i+1:i +block_size+1] for i in ix])

  return x,y


xb,yb = get_batch('train')
xb.shape,yb.shape


for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f"When input is {context.tolist()} the target: {target}")

When input is [24] the target: 43
When input is [24, 43] the target: 58
When input is [24, 43, 58] the target: 5
When input is [24, 43, 58, 5] the target: 57
When input is [24, 43, 58, 5, 57] the target: 1
When input is [24, 43, 58, 5, 57, 1] the target: 46
When input is [24, 43, 58, 5, 57, 1, 46] the target: 43
When input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
When input is [44] the target: 53
When input is [44, 53] the target: 56
When input is [44, 53, 56] the target: 1
When input is [44, 53, 56, 1] the target: 58
When input is [44, 53, 56, 1, 58] the target: 46
When input is [44, 53, 56, 1, 58, 46] the target: 39
When input is [44, 53, 56, 1, 58, 46, 39] the target: 58
When input is [44, 53, 56, 1, 58, 46, 39, 58] the target: 1
When input is [52] the target: 58
When input is [52, 58] the target: 1
When input is [52, 58, 1] the target: 58
When input is [52, 58, 1, 58] the target: 46
When input is [52, 58, 1, 58, 46] the target: 39
When input is [52, 58, 1, 58, 46, 39] the t

In [11]:
class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size) -> None:
    super().__init__()

    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

  def forward(self,idx,targets=None):
    logits = self.token_embedding_table(idx) #idx=(4,8) ...logits get (4,8,vocab_size)

    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C) #reshaped logits -> one row per prediction for cross entropy (32,V)
      targets = targets.view(-1) #fit accordingly
      loss = F.cross_entropy(logits,targets)
    return logits,loss


  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      logits,loss = self(idx)
      # selectss the  predictions at the final time step (we want to generate new targets for the earlier position)
      logits = logits[:,-1,:] #becomes (B,C) -> takes all dim from first and second and the final char of the current s
      probs = F.softmax(logits,dim=1) #converts those scoress into prob that sums to 1
      idx_next = torch.multinomial(probs,num_samples=1) #samples a token ussing those prob
      idx = torch.cat((idx,idx_next),dim=1)#appends the sampled token to the existiing sequence

    return idx

m = BigramLanguageModel(vocab_size)
logits,loss=m(xb,yb)
final = m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)#fun fact: gpt uses the same overall generate one token at a tme proces
print(decode(final[0].tolist()))


lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


In [12]:
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [13]:
batch_size=32
for steps in range(10000):
  xb,yb=get_batch('train')
  logits,loss = m(xb,yb) #forward pass
  optimizer.zero_grad(set_to_none=True) #sets to zero gradient
  loss.backward() #backward pass
  optimizer.step()

print(loss.item())

2.362441062927246


In [14]:
ans = m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=500)
# ans[0].tolist()

In [15]:
print(decode(m.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=500)[0].tolist()))


D:

A:

Ith'dwitile w? beren to'd ff a atrts brey s

ESesenther:
Ithon f at par,
NTmamy an flictong m




Mameld h che IN: an y is aslo'daDut, t thethiceve fur t anowik
Wirghe f bot-d at'prd
Anoper sof usy be, d s me cks bity.
Cis:
INILou f lendys.
Y anditont avenghe m, gs gl tis y.
Wie gh-mmo hizy s m, f lourachigethuiclotif qDWeZPld:
LOubour Witamul we thiech l lisowarrew bland cedanidate, fafive withe thiulsosthis thatwancaurind th'gonimake

S oveprene?
HRar oumnanoupamak in:

The!
The f d sd


In [16]:
# Fake data
torch.manual_seed(1337)
B,T,C=4,8,2 #C =2  features per position
x = torch.randn(B,T,C) #B=4,T=8 positions per sequence, C=2 features per position
x

tensor([[[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]],

        [[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910],
         [ 0.1260, -1.5627],
         [-1.1601, -0.3348],
         [ 0.4478, -0.8016],
         [ 1.5236,  2.5086]],

        [[-0.6631, -0.2513],
         [ 1.0101,  0.1215],
         [ 0.1584,  1.1340],
         [-1.1539, -0.2984],
         [-0.5075, -0.9239],
         [ 0.5467, -1.4948],
         [-1.2057,  0.5718],
         [-0.5974, -0.6937]],

        [[ 1.6455, -0.8030],
         [ 1.3514, -0.2759],
         [-1.5108,  2.1048],
         [ 2.7630, -1.7465],
         [ 1.4516, -1.5103],
         [ 0.8212, -0.2115],
         [ 0.7789,  1.5333],
         [ 1.6097, -0.4032]]])

In [18]:
# Average all earlier position
# Information at each position, incorporates the prev positions
xbow = torch.zeros((B,T,C))
# at position 3 , take the avg of positions 0,1,2 and 3..at pos 4 ,avg positions 0 through 4
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] #if t=2, this selects first three positions
    # print(xprev)
    xbow[b,t] =  torch.mean(xprev,0) #average across the position dim, producing one vector of shape (C,)
    print(xbow)

x[0],xbow[0]

tensor([[[ 0.1808, -0.0700],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000],
         [ 0.0000,  0.0000]]])
tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
      

(tensor([[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [19]:
# Causal averaging without python loops
wei = torch.tril(torch.ones(T,T))
wei = wei/wei.sum(1,keepdim=True)
# wei.shape,x.shape
xbow2 = wei@x #(T,T)@(B,T,C) -> (B,T,C) i.e [8,8] @[4,8,2] -> [4,8,2]
# For each batch and position, the corresponding row of wei combines the features from the allowed positions...
torch.allclose(xbow,xbow2)

False

In [20]:
# Final output , equal weightss for all tokens..0 for future tokens
tril = torch.tril(torch.ones(T,T)) # Causal mask , whcih tokens allowed to talk to each other. 1=allowed, 0=not allowed
# tril
wei = torch.zeros((T,T))#represents the affinity scores(how much attention token a should pay to token B). Before model learns anything,it assumess all past tokenss are equally interesting
wei = wei.masked_fill(tril==0,float('-inf')) #replaces 0 with -inf and 1 with 0's..-inf bcz we want to completely blind the model to future tokens
wei = F.softmax(wei,dim=-1) #applies ssoftmax to each rows
xbow3 = wei@x
# wei.shape,x.shape  [8,8] x [4,8,2] pytorch makess it [4,8,8]@[4,8,2] => rowsxcolumn .column of the first(8) must matcch the second' row i.e 8
torch.allclose(xbow2,xbow3), torch.allclose(xbow,xbow3)

(True, False)

In [21]:
# Self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)
head_size=16

key = nn.Linear(C,head_size,bias=False) #learned linear transformation fro 32 features to 16 features
query  = nn.Linear(C,head_size,bias=False)
value = nn.Linear(C,head_size,bias=False)

k = key(x)
q = query(x)
v = value(x)

# k.shape,q.shape,v.shape
# q.shape,k.shape,k.transpose(-2,-1).shape
wei = q@k.transpose(-2,-1)
wei.shape #the result is a score matrix for each sequence ...

torch.Size([4, 8, 8])

In [22]:
# Scaling the scores
wei = q@k.transpose(-2,-1)*(head_size**-0.5) #scores = QK.T/math.sqrt(dk) where dk is the query/key dim

In [24]:
# Applying the causal mask
tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril==0,float('-inf')) #with four positions, the attention weights after masking and softmax will have this pattern
wei = F.softmax(wei,dim=-1) #we normalize over the key positionss, separately for each query position
wei.shape
v = value(x)
out = wei@v
wei.shape,v.shape,out.shape

(torch.Size([4, 8, 8]), torch.Size([4, 8, 16]), torch.Size([4, 8, 16]))

In [ ]:
# Entire computation performed by open attention head
# Input x(B,T,C) -> Q(Query),K(Key),V(Value) -> QxK.t/math.sqrt Calculate attention scores: (B,T,T) -> Causal mask (Softmax) turn scores into weights -> weights x V. (Output: B,T,head_size)

In [25]:
# Variables configure the model. In particular , each head uses 384//6 = 64 features
import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size=64
block_size=256
max_iters = 500
eval_interval=500
learning_rate=3e-4
eval_iters=50

device = 'cuda' if torch.cuda.is_available() else 'cpu'

n_embd = 384
n_head=6
n_layer=6
dropout=0.2
torch.manual_seed(1337)

device

'cuda'

In [26]:
# Single attention Head
class Head(nn.Module):
  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embd,head_size,bias=False)
    self.query = nn.Linear(n_embd,head_size,bias=False)
    self.value = nn.Linear(n_embd,head_size,bias=False)

    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    self.dropout = nn.Dropout(dropout)


  def forward(self,x):
    B,T,C  = x.shape
    k = self.key(x)
    q = self.query(x)
    # Scaled query-key scores
    wei  = q@k.transpose(-2,-1)*(k.size(-1)**-0.5)

    # Prevent looking at future positions
    wei = wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))

    # Convert scores into attention weightss
    wei = F.softmax(wei,dim=-1)
    wei = self.dropout(wei)

    # Combine value vectors
    v = self.value(x)
    out = wei@v
    return out

In [27]:
# Multi-Head Attention
#One attention head learnss one set of ways to match and combne positions, multiple heads let the model learnn several patterns in parallel
class MultiHeadAttention(nn.Module):
  def __init__(self,num_heads,head_size):
    super().__init__()
    self.heads =  nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd,n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    out = torch.cat([h(x) for h in self.heads],dim=-1)
    out = self.proj(out)
    out = self.dropout(out)
    return out


In [28]:
# Expands the feature dimension, applies a non-linearity and projects back to the embedding dim
class FeedForward(nn.Module):
  def __init__(self,n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd,4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd,n_embd),
        nn.Dropout(dropout)
    )

  def forward(self,x):
    return self.net(x)

In [29]:
# Each block performs attention first,then the feed forward network..Both're wrapped with residual connections and preced by LayerNorm
class Block(nn.Module):
  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size = n_embd//n_head
    self.sa = MultiHeadAttention(n_head,head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self,x):
    x = x+self.sa(self.ln1(x))
    x = x+self.ffwd(self.ln2(x))
    return x


In [30]:
# Complete Bigram Model
class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.position_embedding_table = nn.Embedding(block_size,n_embd)
    self.token_embedding_table  = nn.Embedding(vocab_size,n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd,n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd,vocab_size)

  def forward(self,idx,targets=None):
    B,T = idx.shape
    pos_emb = self.position_embedding_table(torch.arange(T,device=idx.device))
    token_emb = self.token_embedding_table(idx)
    x = token_emb  + pos_emb
    x = self.blocks(x)
    x = self.ln_f(x)
    logits= self.lm_head(x)
    if targets is None:
      loss = None
    else:
      B,T,C=logits.shape
      logits = logits.reshape(B*T,C)
      targets = targets.reshape(B*T)
      loss = F.cross_entropy(logits,targets)
    return logits,loss

  @torch.no_grad()
  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:,-block_size:]
      logits,loss = self(idx_cond)
      # Generation only needss the final position
      if loss is None:
        logits = logits[:,-1,:]

      probs = F.softmax(logits,dim=-1)
      idx_next = torch.multinomial(probs,num_samples=1)


In [34]:
model = BigramLanguageModel(vocab_size)
model.to(device)
print(sum(p.numel() for p in model.parameters())/1e6,'M Parameters')
optimizer = torch.optim.AdamW(model.parameters(),lr=learning_rate)
for iter in range(max_iters):
  xb,yb = get_batch('train')
  xb=xb.to(device)
  yb=yb.to(device)
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
  if iter % eval_interval == 0:
        print(f"step {iter}: train loss {loss.item():.4f}")

10.788929 M Parameters
step 0: train loss 4.3807


KeyboardInterrupt: 

In [35]:
import torch.amp

# 1. Compile the model (PyTorch 2.0+) for fused kernels and graph optimization
# Note: This might take a minute or two to warm up during the first few steps.
compiled_model = torch.compile(model)

# 2. Initialize Gradient Scaler for Mixed Precision (FP16)
scaler = torch.amp.GradScaler('cuda')

optimizer = torch.optim.AdamW(compiled_model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    # Fetch batch
    xb, yb = get_batch('train')
    xb = xb.to(device, non_blocking=True)
    yb = yb.to(device, non_blocking=True)

    # 3. Autocast forward pass to FP16
    with torch.amp.autocast('cuda', dtype=torch.float16):
        logits, loss = compiled_model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    # 4. Backward pass with scaled loss to prevent underflow
    scaler.scale(loss).backward()

    # 5. Step optimizer and update scale
    scaler.step(optimizer)
    scaler.update()

    if iter % eval_interval == 0:
        print(f"step {iter}: train loss {loss.item():.4f}")

W1015 06:13:14.855000 716 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


step 0: train loss 2.4704
